# Safe zero at 0.02 V/s — Unit 2, DAC 5

Visible physical preflight, both-polarity timing/slope/zero-code/latch tests and network rearm. Outputs are isolated. The Keysight is on global DAC 5. Unit 1 has USB power only and is observed without output/recovery commands. Absolute zero accuracy retains ±0.5 mV and is reported independently of rate. Calibration is not changed.


In [ ]:
RUN_LIVE=False
ALL_OUTPUTS_ISOLATED=False
CHANNEL=5
BASE='http://127.0.0.1:8765'
TOLERANCE_V=.0005


In [ ]:
PREFLIGHT_PASSED=False
import json,time,uuid,re,math,statistics,csv
from pathlib import Path
from datetime import datetime,timezone
from urllib.request import Request,urlopen
import matplotlib.pyplot as plt
assert RUN_LIVE and ALL_OUTPUTS_ISOLATED,'Enable both live-test flags first'
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'malab/dashboard').is_dir())
results=root/'malab/notebooks/results'/('modules_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
results.mkdir(parents=True)
token=re.search(r'name="dac-token" content="([^"]+)"',urlopen(BASE).read().decode()).group(1)
def state():return json.load(urlopen(BASE+'/api/state',timeout=3))
s=state();assert s['meter']['channel']==CHANNEL,'Meter label and physical connection must match'
UNIT=(CHANNEL-1)//4+1
assert UNIT==2 and CHANNEL==5
from malab.bench_config import load_bench_config
assert s["units"][UNIT-1]["address"]==load_bench_config()["units"][UNIT-1]["address"]
CHANNELS=list(range(4*(UNIT-1)+1,4*UNIT+1))
def row(s,ch=CHANNEL):return s['channels'][ch-1]
def unit(s):return s['units'][UNIT-1]
def codes(s):return [row(s,ch)['code'] for ch in CHANNELS]
def compact(s):return {k:s[k] for k in ('now','units','channels','meter')}
def wait_for(predicate,timeout=25):
    deadline=time.monotonic()+timeout
    while time.monotonic()<deadline:
        s=state()
        if predicate(s):return s
        time.sleep(.05)
    raise AssertionError('State deadline expired: '+json.dumps(unit(s)))
def action(name,**fields):
    payload=dict(action=name,unit=UNIT,request_id=str(uuid.uuid4()),**fields)
    request=Request(BASE+'/api/action',json.dumps(payload).encode(),headers={'Content-Type':'application/json','X-DAC-Token':token})
    job=json.load(urlopen(request,timeout=7))
    s=wait_for(lambda s:s['jobs'].get(job['id'],{}).get('state') in ('complete','failed'))
    job=s['jobs'][job['id']]
    assert job['state']=='complete',job
    return job
def fresh(s):
    assert unit(s)['network']['state']=='good',unit(s)
    assert s['now']-unit(s)['network']['last_ok']<2.5
    assert s['meter']['state']=='good' and s['now']-s['meter']['at']<2.5
    assert not unit(s)['device_error'],unit(s)['device_error']
def readings(count=5):
    samples=[];previous=None
    while len(samples)<count:
        s=state();fresh(s)
        if s['meter']['at']!=previous:samples.append(compact(s));previous=s['meter']['at']
        time.sleep(.04)
    return samples
def save(name,value):(results/(name+'.json')).write_text(json.dumps(value,indent=2),encoding='utf-8')
wait_for(lambda s:unit(s)['network']['state']=='good' and s['meter']['state']=='good')
action('initialize');time.sleep(.4)
initial=state();fresh(initial);save('initial',compact(initial))
profile=row(initial)['calibration'];STEP=profile['volts_per_code'];OFFSET=profile['offset_nanovolts']*1e-9
ZERO_CODES=[row(initial,ch)['calibration']['zero_code'] for ch in CHANNELS]
def code_for(v):return min(262143,max(0,int(math.floor((v-OFFSET)/STEP+.5))))
print('Results:',results,'channel:',CHANNEL,'unit:',UNIT,'profile:',profile)

# Validate the physical meter association before the wider sweep.
probe=[]
for target in [0,.05,0]:
    action('set',channel=CHANNEL,voltage=target);time.sleep(.4)
    probe.append(dict(target=target,meter=statistics.mean(s['meter']['voltage'] for s in readings())))
save('preflight',probe)
assert abs((probe[1]['meter']-probe[0]['meter'])-.05)<.005,probe
assert abs(probe[2]['meter']-probe[0]['meter'])<.0005,probe
PREFLIGHT_PASSED=True
print('PASS: meter follows DAC 5 small step and returns; readings:',probe)


## S5 — Safe zero while TCP is disconnected; absolute zero checked separately

In [ ]:
assert PREFLIGHT_PASSED,'DAC 5 physical preflight has not passed'
safe_rows=[]
for sign in [1,-1]:
    for ch,v in zip([ch for ch in CHANNELS if ch!=CHANNEL],[.01,-.01,.02]):action('set',channel=ch,voltage=v)
    action('set',channel=CHANNEL,voltage=sign*.05);time.sleep(.4)
    before=state();action('disconnect')
    started=time.time();action('safe_zero');trace=[];deadline=time.monotonic()+10
    while time.monotonic()<deadline:
        s=state();trace.append(compact(s))
        assert unit(s)['network']['state']!='good'
        assert unit(s)['recovery']['state'] in ('ACTIVE','LATCHED')
        if unit(s)['recovery']['state']=='LATCHED':break
        time.sleep(.05)
    duration=time.time()-started
    assert unit(trace[-1])['recovery']['state']=='LATCHED'
    assert 2.4<=duration<=3.3,duration
    assert codes(trace[-1])==ZERO_CODES
    active=[s for s in trace if unit(s)['recovery']['state']=='ACTIVE' and .005<abs(row(s)['voltage'])<.045]
    points=sorted({s['meter']['at']:s['meter']['voltage'] for s in active}.items());assert len(points)>8
    tm=statistics.mean(t for t,v in points);vm=statistics.mean(v for t,v in points)
    slope=sum((t-tm)*(v-vm) for t,v in points)/sum((t-tm)**2 for t,v in points)
    assert .019<=abs(slope)<=.0202 and sign*slope<0,slope
    time.sleep(.4);latched=state();assert unit(latched)['recovery']['state']=='LATCHED'
    result=dict(sign=sign,duration_s=duration,slope_V_per_s=slope,meter_zero=latched['meter']['voltage'],samples=trace)
    safe_rows.append(result);save('safe_zero',safe_rows);print({k:v for k,v in result.items() if k!='samples'})
    action('network');rearmed=wait_for(lambda s:unit(s)['network']['state']=='good' and unit(s)['recovery']['state']=='READY')
    assert codes(rearmed)==ZERO_CODES
print('PASS: all four stored zero codes, TCP-independent ramp, measured slew, latch and rearm')


In [ ]:
assert max(abs(r['meter_zero']) for r in safe_rows)<=TOLERANCE_V,'Safe-zero analog endpoint exceeds ±0.5 mV'
print('PASS: absolute safe-zero endpoint accuracy')


In [ ]:
assert PREFLIGHT_PASSED
for ch in CHANNELS:action('set',channel=ch,voltage=0)
time.sleep(.4);final=state();fresh(final);save('final',compact(final))
assert codes(final)==ZERO_CODES
assert all(job['unit']==UNIT for job in final['jobs'].values())
print('Final Unit 2 codes:',codes(final),'meter:',final['meter']['voltage'])
